# LeetCode 3455 — Shortest Matching Substring bằng Python

- Link: https://leetcode.com/problems/shortest-matching-substring/
- Độ khó: **Hard**
- Chủ đề: **Tách mẫu `A*B*C`**, tìm **mọi vị trí xuất hiện** của từng đoạn bằng **KMP** (hoặc hàm Z), ghép bằng **hai con trỏ** / `bisect`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def shortestMatchingSubstring(self, s: str, p: str) -> int:
```

## 1. Mục tiêu học

1. Hiểu mẫu có đúng hai dấu `*` ⇔ `p = A*B*C`; một xâu con khớp ⇔ nó **bắt đầu bằng `A`**, **kết thúc bằng `C`**, và **chứa `B`** ở giữa, ba phần **không chồng lên nhau**.
2. Liệt kê mọi vị trí xuất hiện (kể cả chồng nhau) của một mẫu trong `O(n + m)` bằng **KMP**; xử lý mẫu rỗng.
3. Với mỗi lần xuất hiện của `B`: chọn `A` **muộn nhất** kết thúc trước nó và `C` **sớm nhất** bắt đầu sau nó ⇒ hai con trỏ.
4. Cách thay thế: với mỗi lần xuất hiện của `A`, chọn tham lam `B` và `C` sớm nhất bằng `bisect`.
5. Phân biệt "khớp sớm nhất bên trái" (như `re.search`) với "khớp **ngắn nhất**".

## 2. Đề bài & ví dụ

Cho `s` và mẫu `p` chứa **đúng hai** ký tự `*` (mỗi `*` khớp dãy bất kỳ, kể cả rỗng). Trả về độ dài **xâu con ngắn nhất** của `s` khớp `p`; không có ⇒ `-1`. Xâu con rỗng được tính.

**Ví dụ 1:** `s = "abaacbaecebce", p = "ba*c*ce"` → `8` (`"baecebce"`).

**Ví dụ 2:** `s = "baccbaadbc", p = "cc*baa*adb"` → `-1`.

**Ví dụ 3:** `s = "a", p = "**"` → `0`.

**Ví dụ 4:** `s = "madlogic", p = "*adlogi*"` → `6`.

**Ràng buộc:** `1 ≤ |s|, |p| ≤ 10^5`.

In [1]:
examples = [("abaacbaecebce", "ba*c*ce", 8), ("baccbaadbc", "cc*baa*adb", -1),
            ("a", "**", 0), ("madlogic", "*adlogi*", 6)]
for s, p, exp in examples:
    print(repr(s), repr(p), '->', p.split('*'), '| kỳ vọng', exp)

'abaacbaecebce' 'ba*c*ce' -> ['ba', 'c', 'ce'] | kỳ vọng 8
'baccbaadbc' 'cc*baa*adb' -> ['cc', 'baa', 'adb'] | kỳ vọng -1
'a' '**' -> ['', '', ''] | kỳ vọng 0
'madlogic' '*adlogi*' -> ['', 'adlogi', ''] | kỳ vọng 6


## 3. Kiến thức nền

### 3.1. Khớp một xâu với `A*B*C`

Xâu `t` khớp `A*B*C` ⇔ `t = A + x + B + y + C` với `x, y` bất kỳ ⇔ `t` bắt đầu bằng `A`, kết thúc bằng `C`, và trong phần giữa (sau `A`, trước `C`) có một lần xuất hiện của `B`.

In [2]:
import re

def matches(t, p):
    A, B, C = p.split('*')
    if len(t) < len(A) + len(B) + len(C) or not t.startswith(A) or not t.endswith(C):
        return False
    return B in t[len(A):len(t) - len(C)]

rx = lambda p: re.compile(re.escape(p).replace(r'\*', '.*'), re.S)
for t in ["baecebce", "bace", "bacce", "baecebc"]:
    print(t, matches(t, "ba*c*ce"), bool(rx("ba*c*ce").fullmatch(t)))

baecebce True True
bace False False
bacce True True
baecebc False False


### 3.2. KMP — mọi vị trí xuất hiện (kể cả chồng nhau)

Hàm tiền tố `pi` của mẫu; quét văn bản, mỗi khi khớp đủ thì ghi vị trí bắt đầu và lùi `k = pi[k − 1]` để bắt được lần chồng tiếp theo. **Mẫu rỗng** xuất hiện ở mọi vị trí `0..n`.

In [3]:
def kmp_occ(t, pat):
    if not pat:
        return list(range(len(t) + 1))
    m = len(pat)
    pi = [0] * m
    for i in range(1, m):
        k = pi[i - 1]
        while k and pat[i] != pat[k]:
            k = pi[k - 1]
        if pat[i] == pat[k]:
            k += 1
        pi[i] = k
    res, k = [], 0
    for i, ch in enumerate(t):
        while k and ch != pat[k]:
            k = pi[k - 1]
        if ch == pat[k]:
            k += 1
        if k == m:
            res.append(i - m + 1)
            k = pi[k - 1]
    return res

print(kmp_occ("aaaa", "aa"), kmp_occ("abaacbaecebce", "ce"), kmp_occ("abc", ""))

[0, 1, 2] [8, 11] [0, 1, 2, 3]


### 3.3. Hàm Z — cách khác để tìm vị trí xuất hiện

Ghép `pat + '#' + t`, tính `z`; vị trí `i` trong `t` là lần xuất hiện ⇔ `z[|pat| + 1 + i] ≥ |pat|`.

In [4]:
def z_function(s):
    n = len(s)
    z = [0] * n
    if n:
        z[0] = n
    l = r = 0
    for i in range(1, n):
        if i < r:
            z[i] = min(r - i, z[i - l])
        while i + z[i] < n and s[z[i]] == s[i + z[i]]:
            z[i] += 1
        if i + z[i] > r:
            l, r = i, i + z[i]
    return z

def z_occ(t, pat):
    if not pat:
        return list(range(len(t) + 1))
    z = z_function(pat + '\x00' + t)
    m = len(pat)
    return [i for i in range(len(t) - m + 1) if z[m + 1 + i] >= m]

print(z_occ("aaaa", "aa"), z_occ("abaacbaecebce", "ce"))

[0, 1, 2] [8, 11]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `A = "ba"`, `B = "c"`, `C = "ce"`, `s = "abaacbaecebce"` (chỉ số 0..12).

- `A` xuất hiện tại `1, 5`; `B` tại `4, 8, 11`; `C` tại `8, 11`.
- Lấy `B` tại `8`: `A` muộn nhất kết thúc ≤ 8 là `5` (kết thúc 7); `C` sớm nhất bắt đầu ≥ `8 + 1 = 9` là `11` ⇒ xâu `s[5..12]` dài `11 + 2 − 5 = 8`.
- Lấy `B` tại `4`: `A` tại `1`; `C` ≥ 5 là `8` ⇒ `8 + 2 − 1 = 9`.

Ngắn nhất `8`. Nhận xét: cố định `B`, muốn đoạn ngắn thì `A` càng **muộn** càng tốt, `C` càng **sớm** càng tốt — hai lựa chọn độc lập.

In [5]:
s, p = examples[0][:2]
A, B, C = p.split('*')
print('A:', kmp_occ(s, A), '| B:', kmp_occ(s, B), '| C:', kmp_occ(s, C))

A: [1, 5] | B: [4, 8, 11] | C: [8, 11]


## 5. Cách 1 — Brute Force: thử mọi xâu con

In [6]:
def shortest_brute(s, p):
    best = -1
    for i in range(len(s) + 1):
        for j in range(i, len(s) + 1):
            if matches(s[i:j], p) and (best == -1 or j - i < best):
                best = j - i
    return best

for s, p, exp in examples:
    print(shortest_brute(s, p), '| kỳ vọng', exp)

8 | kỳ vọng 8
-1 | kỳ vọng -1
0 | kỳ vọng 0
6 | kỳ vọng 6


**Độ phức tạp:** `O(n²)` xâu con × `O(n)` kiểm tra ⇒ `O(n³)`.

## 6. Ý tưởng thuật toán / bước đột phá

1. `A, B, C = p.split('*')`; tìm danh sách vị trí `oa, ob, oc` (tăng dần).
2. Duyệt `j ∈ ob` tăng dần. Con trỏ `i` trên `oa`: tiến khi `oa[i] + |A| ≤ j` ⇒ `oa[i − 1]` là `A` muộn nhất hợp lệ. Con trỏ `k` trên `oc`: tiến khi `oc[k] < j + |B|` ⇒ `oc[k]` là `C` sớm nhất hợp lệ.
3. Ứng viên `oc[k] + |C| − oa[i − 1]`.

Cả hai con trỏ chỉ tiến ⇒ `O(|oa| + |ob| + |oc|)`. Cộng KMP `O(n + |p|)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Cố định `A`, tham lam chọn `B` và `C` sớm nhất (`bisect`)

Với `A` tại `i`: `B` sớm nhất bắt đầu ≥ `i + |A|`, rồi `C` sớm nhất bắt đầu ≥ (vị trí `B`) `+ |B|`. Chọn `B` sớm nhất không làm `C` muộn hơn ⇒ tham lam đúng với `A` cố định.

In [7]:
import bisect

def shortest_per_A(s, p):
    A, B, C = p.split('*')
    a, b, c = len(A), len(B), len(C)
    oa, ob, oc = kmp_occ(s, A), kmp_occ(s, B), kmp_occ(s, C)
    best = -1
    for i in oa:
        x = bisect.bisect_left(ob, i + a)
        if x == len(ob):
            break                        # A muộn hơn càng không có B
        j = ob[x]
        y = bisect.bisect_left(oc, j + b)
        if y == len(oc):
            break
        L = oc[y] + c - i
        if best == -1 or L < best:
            best = L
    return best

for s, p, exp in examples:
    print(shortest_per_A(s, p), '| kỳ vọng', exp)

8 | kỳ vọng 8
-1 | kỳ vọng -1
0 | kỳ vọng 0
6 | kỳ vọng 6


### 7.2. Cách 3 — Duyệt `B`, hai con trỏ cho `A` và `C` (lời giải chính)

In [8]:
def shortest_two_ptr(s, p, occ=kmp_occ):
    A, B, C = p.split('*')
    a, b, c = len(A), len(B), len(C)
    oa, ob, oc = occ(s, A), occ(s, B), occ(s, C)
    best, i, k = -1, 0, 0
    for j in ob:
        while i < len(oa) and oa[i] + a <= j:
            i += 1
        if i == 0:
            continue                     # chưa có A nào kết thúc trước B này
        while k < len(oc) and oc[k] < j + b:
            k += 1
        if k == len(oc):
            break                        # B muộn hơn càng không có C
        L = oc[k] + c - oa[i - 1]
        if best == -1 or L < best:
            best = L
    return best

for s, p, exp in examples:
    print(shortest_two_ptr(s, p), '| kỳ vọng', exp)

8 | kỳ vọng 8
-1 | kỳ vọng -1
0 | kỳ vọng 0
6 | kỳ vọng 6


### 7.3. Cách 4 — Như cách 3 nhưng tìm vị trí bằng hàm Z

In [9]:
def shortest_z(s, p):
    return shortest_two_ptr(s, p, occ=z_occ)

for s, p, exp in examples:
    print(shortest_z(s, p), '| kỳ vọng', exp)

8 | kỳ vọng 8
-1 | kỳ vọng -1
0 | kỳ vọng 0
6 | kỳ vọng 6


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1.

In [10]:
def trace_two_ptr(s, p):
    A, B, C = p.split('*')
    a, b, c = len(A), len(B), len(C)
    oa, ob, oc = kmp_occ(s, A), kmp_occ(s, B), kmp_occ(s, C)
    print(f"A={A!r} tại {oa}, B={B!r} tại {ob}, C={C!r} tại {oc}")
    best, i, k = -1, 0, 0
    for j in ob:
        while i < len(oa) and oa[i] + a <= j:
            i += 1
        if i == 0:
            print(f"B tại {j}: chưa có A kết thúc trước -> bỏ qua"); continue
        while k < len(oc) and oc[k] < j + b:
            k += 1
        if k == len(oc):
            print(f"B tại {j}: không còn C phía sau -> dừng"); break
        L = oc[k] + c - oa[i - 1]
        best = L if best == -1 else min(best, L)
        print(f"B tại {j}: A muộn nhất {oa[i - 1]}, C sớm nhất {oc[k]} -> xâu {s[oa[i - 1]:oc[k] + c]!r} dài {L}")
    print('kết quả:', best)

trace_two_ptr(*examples[0][:2])

A='ba' tại [1, 5], B='c' tại [4, 8, 11], C='ce' tại [8, 11]
B tại 4: A muộn nhất 1, C sớm nhất 8 -> xâu 'baacbaece' dài 9
B tại 8: A muộn nhất 5, C sớm nhất 11 -> xâu 'baecebce' dài 8
B tại 11: không còn C phía sau -> dừng
kết quả: 8


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def shortestMatchingSubstring(self, s: str, p: str) -> int:
        def occ(t, pat):
            if not pat:
                return list(range(len(t) + 1))
            m = len(pat)
            pi = [0] * m
            for i in range(1, m):
                k = pi[i - 1]
                while k and pat[i] != pat[k]:
                    k = pi[k - 1]
                if pat[i] == pat[k]:
                    k += 1
                pi[i] = k
            res, k = [], 0
            for i, ch in enumerate(t):
                while k and ch != pat[k]:
                    k = pi[k - 1]
                if ch == pat[k]:
                    k += 1
                if k == m:
                    res.append(i - m + 1)
                    k = pi[k - 1]
            return res

        A, B, C = p.split('*')
        a, b, c = len(A), len(B), len(C)
        oa, ob, oc = occ(s, A), occ(s, B), occ(s, C)
        best, i, k = -1, 0, 0
        for j in ob:
            while i < len(oa) and oa[i] + a <= j:
                i += 1
            if i == 0:
                continue
            while k < len(oc) and oc[k] < j + b:
                k += 1
            if k == len(oc):
                break
            L = oc[k] + c - oa[i - 1]
            if best == -1 or L < best:
                best = L
        return best

for s, p, exp in examples:
    got = Solution().shortestMatchingSubstring(s, p)
    print(got, 'OK' if got == exp else 'SAI')

8 OK
-1 OK
0 OK
6 OK


## 10. Vì sao thuật toán đúng

1. **Đặc trưng khớp (3.1):** xâu con `s[l..r)` khớp ⇔ tồn tại vị trí `i = l` của `A`, `j` của `B`, `k` của `C` với `i + |A| ≤ j`, `j + |B| ≤ k`, `k + |C| = r`.
2. **Cố định `j`:** độ dài `= (k + |C|) − i` nhỏ nhất khi `i` lớn nhất (thỏa `i + |A| ≤ j`) và `k` nhỏ nhất (thỏa `k ≥ j + |B|`); hai điều kiện độc lập.
3. **Hai con trỏ:** khi `j` tăng, tập `A` hợp lệ chỉ **mở rộng** (con trỏ `i` chỉ tiến), tập `C` hợp lệ chỉ **thu hẹp từ trái** (con trỏ `k` chỉ tiến). Nếu với `j` nào đó không còn `C` thì mọi `j` lớn hơn cũng không ⇒ `break` đúng.
4. **Mẫu rỗng:** xuất hiện tại mọi `0..n` nên các ràng buộc vẫn đúng (ví dụ `p = "**"` cho `i = j = k` ⇒ độ dài 0).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi xâu con | `O(n³)` | `O(n)` |
| 2. Cố định `A` + `bisect` | `O(n + |p| + |oa| log n)` | `O(n)` |
| 3. Hai con trỏ (KMP) | `O(n + |p|)` | `O(n)` |
| 4. Hai con trỏ (Z) | `O(n + |p|)` | `O(n + |p|)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'per_A': shortest_per_A, 'two_ptr': shortest_two_ptr, 'z': shortest_z,
         'submit': lambda s, p: Solution().shortestMatchingSubstring(s, p)}
edge = [
    ("abc", "a**", 1), ("abc", "**c", 1), ("abc", "*b*", 1),
    ("abc", "a*b*c", 3), ("abc", "a*c*b", -1),
    ("aaa", "a*a*a", 3), ("aa", "a*a*a", -1),        # ba đoạn không được chồng nhau
    ("xyz", "*q*", -1),
]
for s, p, exp in examples + edge:
    assert shortest_brute(s, p) == exp, (s, p, shortest_brute(s, p))
    for name, f in impls.items():
        assert f(s, p) == exp, (name, s, p)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [13]:
import random
rng = random.Random(3455)
cnt = 0
for _ in range(3000):
    alpha = rng.choice(['ab', 'abc', 'a'])
    s = ''.join(rng.choice(alpha) for _ in range(rng.randint(1, 12)))
    p = '*'.join(''.join(rng.choice(alpha) for _ in range(rng.randint(0, 3))) for _ in range(3))
    exp = shortest_brute(s, p)
    for name, f in impls.items():
        assert f(s, p) == exp, (name, s, p)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên.')

Khớp vét cạn trên 3000 bộ ngẫu nhiên.


### 12.3. Ca lớn `|s| = |p| = 10^5`

In [14]:
import time
n = 10 ** 5
big = {
    'toàn a, mẫu ngắn': ('a' * n, 'aaa*aaaaa*aa'),
    'mẫu dài gần |s|': ('ab' * (n // 2), 'ab' * 20000 + '*' + 'ab' * 10000 + '*' + 'ab' * 15000),
    'ngẫu nhiên ab': (''.join(rng.choice('ab') for _ in range(n)), 'abba*baab*aabb'),
    'không khớp': ('a' * n, 'a*b*a'),
}
for name, (s, p) in big.items():
    t = time.time(); r = Solution().shortestMatchingSubstring(s, p); t1 = time.time() - t
    t = time.time(); r2 = shortest_per_A(s, p); t2 = time.time() - t
    t = time.time(); r3 = shortest_z(s, p); t3 = time.time() - t
    assert r == r2 == r3
    print(f"{name:18s}: {r:6d} | KMP+2 con trỏ {t1:.2f}s | cố định A {t2:.2f}s | Z {t3:.2f}s")

toàn a, mẫu ngắn  :     10 | KMP+2 con trỏ 0.06s | cố định A 0.08s | Z 0.12s
mẫu dài gần |s|   :  90000 | KMP+2 con trỏ 0.03s | cố định A 0.03s | Z 0.11s


ngẫu nhiên ab     :     12 | KMP+2 con trỏ 0.02s | cố định A 0.03s | Z 0.08s
không khớp        :     -1 | KMP+2 con trỏ 0.03s | cố định A 0.02s | Z 0.06s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ lấy lần xuất hiện **đầu tiên** của mỗi đoạn (chuỗi `find`)

In [15]:
def wrong_first_find(s, p):
    A, B, C = p.split('*')
    i = s.find(A)
    if i < 0:
        return -1
    j = s.find(B, i + len(A))
    if j < 0:
        return -1
    k = s.find(C, j + len(B))
    return -1 if k < 0 else k + len(C) - i

s, p, _ = examples[0]
print('sai :', wrong_first_find(s, p), '| đúng:', Solution().shortestMatchingSubstring(s, p))

sai : 9 | đúng: 8


**Vì sao sai:** `find` cho xâu khớp **bắt đầu sớm nhất**, không phải **ngắn nhất**. Ở ví dụ 1, khớp bắt đầu sớm nhất là `s[1:10] = "baacbaece"` (dài 9), còn khớp bắt đầu tại 5 là `"baecebce"` chỉ dài 8.

### Lỗi 2 — Cho phép các đoạn chồng lên nhau (`C` bắt đầu từ vị trí bắt đầu của `B`)

In [16]:
def wrong_overlap(s, p):
    A, B, C = p.split('*')
    a, b, c = len(A), len(B), len(C)
    oa, ob, oc = kmp_occ(s, A), kmp_occ(s, B), kmp_occ(s, C)
    best, i, k = -1, 0, 0
    for j in ob:
        while i < len(oa) and oa[i] <= j:          # SAI: phải là oa[i] + a <= j
            i += 1
        if i == 0:
            continue
        while k < len(oc) and oc[k] < j:            # SAI: phải là oc[k] < j + b
            k += 1
        if k == len(oc):
            break
        L = max(oc[k] + c, j + b) - oa[i - 1]
        if best == -1 or L < best:
            best = L
    return best

print('sai :', wrong_overlap("aa", "a*a*a"), '| đúng:', Solution().shortestMatchingSubstring("aa", "a*a*a"))

sai : 1 | đúng: -1


**Vì sao sai:** `A + x + B + y + C` — ba đoạn nằm **nối tiếp**, không dùng chung ký tự. `"aa"` không thể chứa ba chữ `a` rời nhau.

### Lỗi 3 — Không xử lý đoạn rỗng (KMP trên mẫu rỗng trả về danh sách rỗng)

In [17]:
def kmp_occ_no_empty(t, pat):
    if not pat:
        return []                                   # SAI: mẫu rỗng xuất hiện ở mọi vị trí
    return kmp_occ(t, pat)

for s, p in [("a", "**"), ("madlogic", "*adlogi*")]:
    print(repr(p), 'sai :', shortest_two_ptr(s, p, occ=kmp_occ_no_empty), '| đúng:', Solution().shortestMatchingSubstring(s, p))

'**' sai : -1 | đúng: 0
'*adlogi*' sai : -1 | đúng: 6


**Vì sao sai:** đoạn rỗng (`A`, `B` hoặc `C`) luôn khớp ở mọi vị trí. Ví dụ 3, 4 của đề rơi đúng vào trường hợp này.

### Lỗi 4 — Dùng `re.search` với `.*?` (không tham lam) và tin rằng nó cho xâu ngắn nhất

In [18]:
def wrong_regex(s, p):
    A, B, C = p.split('*')
    m = re.search(re.escape(A) + '.*?' + re.escape(B) + '.*?' + re.escape(C), s)
    return -1 if m is None else m.end() - m.start()

s, p, _ = examples[0]
print('sai :', wrong_regex(s, p), '| đúng:', Solution().shortestMatchingSubstring(s, p))

sai : 9 | đúng: 8


**Vì sao sai:** biểu thức chính quy luôn ưu tiên khớp **bắt đầu sớm nhất** (rồi mới tới độ dài); `.*?` chỉ làm phần giữa ngắn nhất **với điểm bắt đầu đó**. Ngoài ra `re` có thể quay lui rất chậm với dữ liệu xấu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Mẫu với `*` cố định số lượng ⇒ tách thành các đoạn cố định, tìm vị trí từng đoạn.
- "Ngắn nhất chứa các mốc theo thứ tự" ⇒ cố định mốc giữa, chọn mốc trái **muộn nhất**, mốc phải **sớm nhất**.
- Cần **mọi** lần xuất hiện (kể cả chồng) ⇒ KMP / Z, không phải `str.find` một lần.
- Luôn xử lý trường hợp đoạn rỗng.

## 15. Biến thể của bài

1. Mẫu có **một** dấu `*`: `A*C` ⇒ với mỗi `A`, `C` sớm nhất sau nó.
2. Mẫu có **nhiều** dấu `*`: `A_1*A_2*…*A_t` ⇒ DP/tham lam qua từng đoạn: với mỗi lần xuất hiện của `A_1`, nhảy tham lam đoạn sớm nhất tiếp theo (cách 2 tổng quát) — `O(t · |oa| log n)`.
3. Có ký tự `?` (khớp đúng một ký tự) trong các đoạn ⇒ tìm vị trí bằng FFT/bitset thay vì KMP.
4. LC 44 Wildcard Matching — khớp **toàn bộ** chuỗi, DP hoặc tham lam.

In [19]:
# Biến thể 2: nhiều dấu '*' — tham lam từ mỗi lần xuất hiện của đoạn đầu
def shortest_many_stars(s, p):
    parts = p.split('*')
    occs = [kmp_occ(s, q) for q in parts]
    best = -1
    for start in occs[0]:
        pos = start + len(parts[0])
        ok = True
        for q, oq in zip(parts[1:], occs[1:]):
            x = bisect.bisect_left(oq, pos)
            if x == len(oq):
                ok = False; break
            pos = oq[x] + len(q)
        if not ok:
            break
        if best == -1 or pos - start < best:
            best = pos - start
    return best

def brute_many(s, p):
    r = rx(p)
    best = -1
    for i in range(len(s) + 1):
        for j in range(i, len(s) + 1):
            if r.fullmatch(s, i, j) and (best == -1 or j - i < best):
                best = j - i
    return best

rng = random.Random(15)
for _ in range(1500):
    s = ''.join(rng.choice('ab') for _ in range(rng.randint(1, 10)))
    p = '*'.join(''.join(rng.choice('ab') for _ in range(rng.randint(0, 2))) for _ in range(rng.randint(2, 5)))
    assert shortest_many_stars(s, p) == brute_many(s, p), (s, p)
print('Biến thể nhiều dấu * khớp vét cạn:', shortest_many_stars("abaacbaecebce", "b*c*e*c"))

Biến thể nhiều dấu * khớp vét cạn: 7


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi xâu con | `O(n³)` | Oracle |
| 2 | Cố định `A`, tham lam `B`, `C` | `O(n log n)` | Tổng quát hóa tốt (nhiều `*`) |
| 3 | Duyệt `B`, hai con trỏ | `O(n + |p|)` | **Nộp bài** |
| 4 | Như 3, tìm vị trí bằng Z | `O(n + |p|)` | Thay thế KMP |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "aaaa"`, `p = "a*a*"`.

**Bài 2 (tính tay):** `s = "xaybxazbx"`, `p = "x*b*x"`.

**Bài 3 (điền TODO):** viết `shortest_one_star(s, p)` cho mẫu chỉ có **một** `*`.

**Bài 4 (tự viết lại):** trả về **vị trí bắt đầu** của xâu ngắn nhất (nhỏ nhất nếu hòa), hoặc `-1`.

**Bài 5 (mini challenge):** đếm số xâu con (theo cặp chỉ số `(l, r)`) khớp `A*B*C` trong `O(n log n)`.

In [20]:
def shortest_one_star_todo(s, p):
    # TODO
    raise NotImplementedError

def shortest_start_todo(s, p):
    # TODO
    raise NotImplementedError

def count_matching_todo(s, p):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `A = "a"`, `B = "a"`, `C = ""`. (2) Cần hai chữ `a` rời nhau. (3) Độ dài 2.

**Bài 2.** (1) `x` tại 0, 4, 8; `b` tại 3. (2) Với `B` tại 3: `A` muộn nhất ≤ 2 là 0; `C` sớm nhất ≥ 4 là 4. (3) Độ dài `4 + 1 − 0 = 5`.

**Bài 3.** (1) `A*C`. (2) Duyệt `C`, giữ `A` muộn nhất kết thúc trước. (3) Hai con trỏ.

**Bài 4.** (1) Lưu thêm `oa[i − 1]` khi cập nhật. (2) Hòa ⇒ so vị trí. (3) Đối chiếu vét cạn.

**Bài 5.** (1) Cố định `l` là vị trí `A`, tìm `r` nhỏ nhất `r_min(l)` như cách 2. (2) Mọi `r ≥ r_min` mà `s[r − |C|..r)` là `C` đều khớp. (3) Đếm số vị trí `C` có `k ≥ r_min − |C|` bằng `bisect`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
a1 = Solution().shortestMatchingSubstring("aaaa", "a*a*")
a2 = Solution().shortestMatchingSubstring("xaybxazbx", "x*b*x")
assert a1 == shortest_brute("aaaa", "a*a*") == 2 and a2 == shortest_brute("xaybxazbx", "x*b*x") == 5
print('Bài 1:', a1, '| Bài 2:', a2)

# Bài 3
def shortest_one_star(s, p):
    A, C = p.split('*')
    oa, oc = kmp_occ(s, A), kmp_occ(s, C)
    best, i = -1, 0
    for k in oc:
        while i < len(oa) and oa[i] + len(A) <= k:
            i += 1
        if i:
            L = k + len(C) - oa[i - 1]
            best = L if best == -1 else min(best, L)
    return best

rng = random.Random(3)
for _ in range(1500):
    s = ''.join(rng.choice('ab') for _ in range(rng.randint(1, 10)))
    p = ''.join(rng.choice('ab') for _ in range(rng.randint(0, 3))) + '*' + ''.join(rng.choice('ab') for _ in range(rng.randint(0, 3)))
    assert shortest_one_star(s, p) == brute_many(s, p)
print('Bài 3 khớp vét cạn.')

Bài 1: 2 | Bài 2: 5
Bài 3 khớp vét cạn.


In [22]:
# Bài 4, 5
def shortest_start(s, p):
    A, B, C = p.split('*')
    a, b, c = len(A), len(B), len(C)
    oa, ob, oc = kmp_occ(s, A), kmp_occ(s, B), kmp_occ(s, C)
    best = (-1, -1)
    i = k = 0
    for j in ob:
        while i < len(oa) and oa[i] + a <= j:
            i += 1
        if i == 0:
            continue
        while k < len(oc) and oc[k] < j + b:
            k += 1
        if k == len(oc):
            break
        cand = (oc[k] + c - oa[i - 1], oa[i - 1])
        if best[0] == -1 or cand < best:
            best = cand
    if best[0] == -1:
        return -1
    # đoạn tối ưu có thể xuất hiện ở nhiều vị trí bắt đầu: tìm vị trí nhỏ nhất bằng cách 2
    L = best[0]
    for st in oa:
        x = bisect.bisect_left(ob, st + a)
        if x == len(ob):
            break
        y = bisect.bisect_left(oc, ob[x] + b)
        if y < len(oc) and oc[y] + c - st == L:
            return st
    return best[1]

def count_matching(s, p):
    A, B, C = p.split('*')
    a, b, c = len(A), len(B), len(C)
    oa, ob, oc = kmp_occ(s, A), kmp_occ(s, B), kmp_occ(s, C)
    total = 0
    for st in oa:
        x = bisect.bisect_left(ob, st + a)
        if x == len(ob):
            break
        y = bisect.bisect_left(oc, ob[x] + b)
        total += len(oc) - y
    return total

def brute_start_count(s, p):
    r = rx(p)
    best, start, cnt = -1, -1, 0
    for i in range(len(s) + 1):
        for j in range(i, len(s) + 1):
            if r.fullmatch(s, i, j):
                cnt += 1
                if best == -1 or j - i < best:
                    best, start = j - i, i
    return start, cnt

rng = random.Random(5)
for _ in range(1500):
    s = ''.join(rng.choice('ab') for _ in range(rng.randint(1, 10)))
    p = '*'.join(''.join(rng.choice('ab') for _ in range(rng.randint(0, 2))) for _ in range(3))
    st, cnt = brute_start_count(s, p)
    assert shortest_start(s, p) == st and count_matching(s, p) == cnt, (s, p)
print('Bài 4, 5 khớp vét cạn. Ví dụ 1: bắt đầu tại', shortest_start(*examples[0][:2]), ', số xâu con khớp =', count_matching(*examples[0][:2]))

Bài 4, 5 khớp vét cạn. Ví dụ 1: bắt đầu tại 5 , số xâu con khớp = 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
A, B, C = p.split('*')
oa, ob, oc = occ(s, A), occ(s, B), occ(s, C)      # occ('') = 0..n
i = k = 0; best = -1
for j in ob:
    while i < len(oa) and oa[i] + len(A) <= j: i += 1
    if i == 0: continue
    while k < len(oc) and oc[k] < j + len(B): k += 1
    if k == len(oc): break
    best = min(best, oc[k] + len(C) - oa[i-1])
```

**Checklist:**

- [ ] Mọi vị trí xuất hiện (kể cả chồng nhau)?
- [ ] Đoạn rỗng xuất hiện ở mọi vị trí `0..n`?
- [ ] Ba đoạn nối tiếp, không chồng (`oa + |A| ≤ j`, `oc ≥ j + |B|`)?
- [ ] Ngắn nhất, không phải sớm nhất?
- [ ] Không có ⇒ `-1`?